# Agentic Benchmark: Pharma Redistribution Priority Analysis
## Golden Solution (Autonomous Supply Chain Analyst)

**Objective**: Identify the top 5 most critical Pharma items for redistribution to high-efficiency zones using deterministic priority metrics and autonomous validation.

**Headroom Focus**: This benchmark tests the agent's ability to:
- Manage complex multi-step workflows without guidance
- Perform intermediate validation and error handling
- Apply sophisticated tie-breaking rules
- Produce deterministic, evaluable outputs
- Handle edge cases (insufficient items, ties, null values)

## Section 1: Import Required Libraries
Import only pandas, numpy, json, and os. These are the sole allowed external libraries for this autonomous task.

In [ ]:
import pandas as pd
import numpy as np
import json
import os

print("✓ All required libraries imported successfully")

: 

## Section 2: Load and Validate Logistics Dataset
Load 'logistics_dataset.csv' into a pandas DataFrame. Verify file existence and required columns. Initialize debug_checks dictionary.

In [ ]:
# Initialize debug checks dictionary
debug_checks = {
    'data_loaded': False,
    'category_exists': False,
    'no_null_demand': False,
    'priority_non_negative': False
}

# Required columns
REQUIRED_COLUMNS = [
    'item_id', 'category', 'daily_demand', 'item_popularity_score', 
    'stockout_count_last_month', 'zone', 'picking_time_seconds'
]

# Load dataset
try:
    # Check if file exists
    data_file = 'logistics_dataset.csv'
    if not os.path.exists(data_file):
        raise FileNotFoundError(f"File '{data_file}' not found in current directory")
    
    # Load CSV into DataFrame
    logistics_df = pd.read_csv(data_file)
    
    # Verify required columns exist
    missing_columns = [col for col in REQUIRED_COLUMNS if col not in logistics_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")
    
    debug_checks['data_loaded'] = True
    print(f"✓ Dataset loaded successfully. Shape: {logistics_df.shape}")
    print(f"✓ Columns present: {list(logistics_df.columns)}")
    
except Exception as e:
    print(f"✗ Error loading dataset: {e}")
    raise

# Display sample data
print("\nDataset sample (first 3 rows):")
print(logistics_df.head(3))

: 

## Section 3: Filter Pharma Category and Verify Data Integrity
Filter for 'Pharma' items and check for null values in critical demand columns.

In [ ]:
try:
    # Verify category column exists
    if 'category' not in logistics_df.columns:
        raise ValueError("'category' column does not exist in dataset")
    debug_checks['category_exists'] = True
    print("✓ 'category' column verified")
    
    # Filter for Pharma items
    pharma_df = logistics_df[logistics_df['category'] == 'Pharma'].copy()
    
    if len(pharma_df) == 0:
        raise ValueError("No Pharma items found in the dataset")
    
    print(f"✓ Filtered {len(pharma_df)} Pharma items from {len(logistics_df)} total items")
    
    # Check for null values in critical demand columns
    null_demand = pharma_df['daily_demand'].isnull().any()
    null_stockout = pharma_df['stockout_count_last_month'].isnull().any()
    null_popularity = pharma_df['item_popularity_score'].isnull().any()
    
    if null_demand or null_stockout or null_popularity:
        raise ValueError(
            f"Null values found: daily_demand={null_demand}, "
            f"stockout_count_last_month={null_stockout}, "
            f"item_popularity_score={null_popularity}"
        )
    
    debug_checks['no_null_demand'] = True
    print("✓ No null values in critical demand columns")
    print(f"\nPharma dataset shape: {pharma_df.shape}")
    
except Exception as e:
    print(f"✗ Error during Pharma filtering: {e}")
    raise

## Section 4: Calculate Redistribution Priority Metric
Apply the priority formula: Priority = (daily_demand × item_popularity_score) + (stockout_count_last_month × 15.5)

In [ ]:
try:
    # Calculate priority metric for each Pharma item
    pharma_df['priority'] = (
        (pharma_df['daily_demand'] * pharma_df['item_popularity_score']) + 
        (pharma_df['stockout_count_last_month'] * 15.5)
    )
    
    # Verify all priority values are non-negative
    min_priority = pharma_df['priority'].min()
    if min_priority < 0:
        raise ValueError(f"Negative priority values detected: minimum = {min_priority}")
    
    debug_checks['priority_non_negative'] = True
    
    print("✓ Priority metric calculated successfully")
    print(f"  Priority range: [{min_priority:.2f}, {pharma_df['priority'].max():.2f}]")
    print(f"  Mean priority: {pharma_df['priority'].mean():.2f}")
    print(f"  Median priority: {pharma_df['priority'].median():.2f}")
    
except Exception as e:
    print(f"✗ Error calculating priority metric: {e}")
    raise

## Section 5: Identify Critical Items Above 75th Percentile
Filter for items NOT in zone 'A' with priority >= 75th percentile. This identifies items that truly need redistribution.

In [ ]:
try:
    # Calculate 75th percentile of priority
    percentile_75 = pharma_df['priority'].quantile(0.75)
    print(f"✓ 75th percentile of priority: {percentile_75:.2f}")
    
    # Filter for items NOT in zone 'A' with priority >= 75th percentile
    critical_items = pharma_df[
        (pharma_df['priority'] >= percentile_75) & 
        (pharma_df['zone'] != 'A')
    ].copy()
    
    print(f"✓ Found {len(critical_items)} items above 75th percentile NOT in zone 'A'")
    
    # If fewer than 5 items, use all qualifying items; otherwise use top 5
    if len(critical_items) < 5:
        print(f"  Note: Fewer than 5 items found ({len(critical_items)}), will use all qualifying items")
    
    pharma_critical_df = critical_items
    
except Exception as e:
    print(f"✗ Error identifying critical items: {e}")
    raise

## Section 6: Apply Tie-Breaking Rules and Select Top 5
Sort by priority (descending), then by item_id (ascending) for deterministic ordering. Extract top 5 item IDs.

In [ ]:
try:
    # Sort by priority (descending) then by item_id (ascending) for tie-breaking
    pharma_critical_df = pharma_critical_df.sort_values(
        by=['priority', 'item_id'],
        ascending=[False, True]
    )
    
    # Select top 5 (or all if fewer than 5)
    pharma_critical_df = pharma_critical_df.head(5)
    
    # Extract item_id values as list[str]
    priority_item_ids: list[str] = pharma_critical_df['item_id'].astype(str).tolist()
    
    print(f"✓ Top {len(priority_item_ids)} critical Pharma items selected:")
    for idx, (item_id, priority) in enumerate(
        zip(pharma_critical_df['item_id'], pharma_critical_df['priority']), 1
    ):
        print(f"  {idx}. {item_id}: priority = {priority:.2f}")
    
except Exception as e:
    print(f"✗ Error during tie-breaking and selection: {e}")
    raise

## Section 7: Calculate Time Savings for Zone A Redistribution
Compute the average picking time in zone A, then calculate total time savings from moving critical items there.

In [ ]:
try:
    # Calculate average picking time for items in zone A
    zone_a_items = logistics_df[logistics_df['zone'] == 'A']
    
    if len(zone_a_items) == 0:
        raise ValueError("No items found in zone 'A' to calculate baseline")
    
    avg_zone_a_time: float = zone_a_items['picking_time_seconds'].mean()
    print(f"✓ Average picking time in zone A: {avg_zone_a_time:.2f} seconds")
    
    # Calculate individual and total savings
    pharma_critical_df['individual_savings'] = (
        pharma_critical_df['picking_time_seconds'] - avg_zone_a_time
    )
    
    total_savings_seconds: float = pharma_critical_df['individual_savings'].sum()
    total_savings_seconds = round(total_savings_seconds, 2)
    
    print(f"✓ Total time savings from redistribution: {total_savings_seconds:.2f} seconds")
    print(f"\nBreakdown by item:")
    for item_id, current_time, savings in zip(
        pharma_critical_df['item_id'],
        pharma_critical_df['picking_time_seconds'],
        pharma_critical_df['individual_savings']
    ):
        print(f"  {item_id}: {current_time:.0f}s → {savings:.2f}s savings")
    
except Exception as e:
    print(f"✗ Error calculating time savings: {e}")
    raise

## Section 8: Store Debug Checks and Validate Execution
Verify all checks pass. If any check fails, halt execution with clear error messaging.

In [ ]:
print("\n" + "="*60)
print("EXECUTION VERIFICATION")
print("="*60)

# Check all debug checks
all_passed = all(debug_checks.values())

for check_name, check_value in debug_checks.items():
    status = "✓ PASS" if check_value else "✗ FAIL"
    print(f"{status}: {check_name}")

if not all_passed:
    failed_checks = [k for k, v in debug_checks.items() if not v]
    raise AssertionError(f"Execution failed. Failed checks: {failed_checks}")

print("\n✓ All verification checks passed successfully!")
print(f"\nFinal Output Summary:")
print(f"  - Total critical items identified: {len(priority_item_ids)}")
print(f"  - Total time savings: {total_savings_seconds:.2f} seconds")
print(f"  - All debug checks: PASS")

## Section 9: Generate and Export Final Results
Save results to CSV and JSON formats for evaluation. Create output directory if needed.

In [ ]:
try:
    # Ensure output directory exists
    output_dir = '/content'
    os.makedirs(output_dir, exist_ok=True)
    
    # Remove temporary column before saving
    export_df = pharma_critical_df.drop(columns=['individual_savings'], errors='ignore')
    
    # Save pharma_critical_df to CSV
    csv_path = os.path.join(output_dir, 'pharma_audit_results.csv')
    export_df.to_csv(csv_path, index=False)
    print(f"✓ Saved pharma_critical_df to {csv_path}")
    
    # Create final strategy JSON
    final_strategy = {
        'priority_item_ids': priority_item_ids,
        'total_savings_seconds': total_savings_seconds,
        'debug_checks': debug_checks
    }
    
    # Save to JSON
    json_path = os.path.join(output_dir, 'final_strategy.json')
    with open(json_path, 'w') as f:
        json.dump(final_strategy, f, indent=2)
    print(f"✓ Saved final_strategy to {json_path}")
    
    # Verify both files exist
    csv_exists = os.path.exists(csv_path)
    json_exists = os.path.exists(json_path)
    
    if not csv_exists or not json_exists:
        raise FileExistsError(f"File export failed. CSV: {csv_exists}, JSON: {json_exists}")
    
    print("\n" + "="*60)
    print("FINAL STRATEGY OUTPUT")
    print("="*60)
    print(json.dumps(final_strategy, indent=2))
    
except Exception as e:
    print(f"✗ Error generating final results: {e}")
    raise